In [ ]:
pdf_1= "PDF\20230304_dengue_all.pdf"
pdf_2= "PDF\20250220_dengue_all.pdf"

In [ ]:
!pip install pdf2image transformers accelerate qwen-vl-utils torch torchvision pandas

In [ ]:
import os
import json
import re
import pandas as pd
from pdf2image import convert_from_path
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# Initialize the Model (Qwen2-VL-7B-Instruct)
# Using bfloat16 for A100 to maximize performance and save memory
model_id = "Qwen/Qwen3-VL-30B-A3B-Thinking"
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
def extract_date_from_filename(filename):
    """Extracts Year, Month, Date from a filename like 20230304_dengue_all.pdf"""
    basename = os.path.basename(filename)
    match = re.search(r'^(\d{4})(\d{2})(\d{2})', basename)
    if match:
        return match.group(1), match.group(2), match.group(3)
    return None, None, None

def process_single_pdf(pdf_path):
    print(f"Processing: {pdf_path}")
    year, month, day = extract_date_from_filename(pdf_path)
    
    # Convert PDF to Images (300 DPI)
    pages = convert_from_path(pdf_path, dpi=300)
    
    extracted_data = []
    
    for page_num, page_image in enumerate(pages):
        print(f"  Analyzing page {page_num + 1}...")
        
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": page_image},
                    {
                        "type": "text", 
                        "text": """
                        Look at this document. Find the table that shows 'Dengue cases by division in Bangladesh'.
                        Extract the data from this table. 
                        The text is in Bengali. Please translate the division names to English.
                        Output the data strictly as a JSON array of objects with the following keys:
                        - "division": (string) Name of the division
                        - "cases_last_24h": (integer) Cases in the last 24 hours
                        - "total_cases": (integer) Total cases
                        
                        Do not include any markdown formatting, explanations, or other text. Just the JSON array.
                        """
                    }
                ]
            }
        ]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to("cuda")

        # Generate output
        generated_ids = model.generate(**inputs, max_new_tokens=1024)
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]
        
        # Parse JSON
        try:
            # Clean up potential markdown code blocks
            clean_json = output_text.replace('```json', '').replace('```', '').strip()
            page_data = json.loads(clean_json)
            
            if page_data and isinstance(page_data, list):
                for row in page_data:
                    row['source_file'] = os.path.basename(pdf_path)
                    row['year'] = year
                    row['month'] = month
                    row['day'] = day
                extracted_data.extend(page_data)
                print(f"  Successfully extracted {len(page_data)} rows from page {page_num + 1}.")
        except json.JSONDecodeError:
            print(f"  Failed to parse JSON on page {page_num + 1}. Raw output:\n{output_text}")
            
    return extracted_data

In [ ]:
# Process the PDFs and merge into a single DataFrame
# Using the variables from the first cell
pdf_files = [pdf_1, pdf_2] 
all_data = []

for pdf in pdf_files:
    # Ensure the path is absolute or relative to the notebook
    if os.path.exists(pdf):
        data = process_single_pdf(pdf)
        all_data.extend(data)
    else:
        print(f"File not found: {pdf}")

# Create DataFrame and save to CSV
if all_data:
    df = pd.DataFrame(all_data)
    print("\nFinal Extracted Data:")
    display(df.head())
    
    output_csv = "dengue_extracted_data.csv"
    df.to_csv(output_csv, index=False)
    print(f"\nData successfully saved to {output_csv}")
else:
    print("No data was extracted.")